#### Стеганография аудио и видео информации. Лабораторная работа №4
|   Группа          |   ФИО             |   
|   :------------:  |   :------------:  |
|   М092501(71)     |   Шарибжанов И.Т. |

**Цель работы:**
Изучение метода скрытия данных в пространственной области методом замены наименее значащего бита, получение навыков реализации изученного метода в программной среде

##### Импорт библиотек и вспомогательные функции

In [19]:
from PIL import Image
import numpy as np

def text_to_bits(text):
    """Перевод строки (UTF-8) в биты"""
    bytes_data = text.encode('utf-8')
    return ''.join(format(b, '08b') for b in bytes_data)

def bits_to_text(bits):
    """Перевод битов обратно в строку (UTF-8)"""
    byte_array = bytearray(int(bits[i:i+8], 2) for i in range(0, len(bits), 8))
    return byte_array.decode('utf-8', errors='ignore')

# --- Дополнительные функции для работы с числами (длиной сообщения) ---
def int_to_bits(number, bit_length=32):
    """Перевод целого числа в битовую строку заданной длины"""
    return format(number, f'0{bit_length}b')

def bits_to_int(bits):
    """Перевод битовой строки в целое число"""
    return int(bits, 2)

##### Функция встраивания (Кодер)

In [20]:
def embed_text(image_path, text, output_path):
    """
    Функция для скрытия текста в изображении методом НЗБ.
    """
    # 1. Загружаем изображение-контейнер
    img = Image.open(image_path)
    img_arr = np.array(img)
    
    # Вытягиваем матрицу пикселей в одномерный массив для удобного побитового прохода
    flat_img = img_arr.flatten()

    # 2. Преобразуем скрываемое текстовое сообщение в массив бит
    # Кодируем строку в байты (utf-8), затем каждый байт переводим в 8 бит
    text_bits = text_to_bits(text)

    # Нам нужно знать длину сообщения при извлечении 
    # Выделим первые 32 бита контейнера под запись длины сообщения
    msg_len = len(text_bits)
    len_bits = int_to_bits(msg_len, 32)

    # Итоговая битовая последовательность: [32 бита длины] + [биты самого текста]
    full_bits = len_bits + text_bits

    # Проверка емкости контейнера
    if len(full_bits) > len(flat_img):
        raise ValueError("Ошибка: Объем сообщения превышает размер контейнера!")

    # 3. Замена младших бит изображения битами сообщения
    for i in range(len(full_bits)):
        # Сбрасываем младший бит пикселя в 0 (побитовое И с 254, т.е. 11111110)
        # и прибавляем нужный бит нашего сообщения (побитовое ИЛИ)
        flat_img[i] = (flat_img[i] & 254) | int(full_bits[i])

    # 4. Сохраняем измененное изображение
    stego_img_arr = flat_img.reshape(img_arr.shape)
    stego_img = Image.fromarray(stego_img_arr)
    stego_img.save(output_path, format='BMP')
    
    print(f"Успех! Данные встроены. Файл сохранен как: {output_path}")

##### Функция извлечения (Декодер)

In [21]:
def extract_text(stego_image_path, output_text_path):
    """
    Функция для извлечения скрытого текста из изображения и сохранения в файл.
    """
    # 1. Загружаем стеганоизображение
    img = Image.open(stego_image_path)
    flat_img = np.array(img).flatten()

    # 2. Извлекаем первые 32 бита, чтобы узнать длину скрытого сообщения
    len_bits = ''
    for i in range(32):
        # Получаем младший бит (побитовое И с 1, т.е. с 00000001)
        len_bits += str(flat_img[i] & 1)

    # Переводим из двоичной системы в десятичную
    msg_len = bits_to_int(len_bits)

    # 3. Извлекаем биты самого текстового сообщения
    text_bits = ''
    for i in range(32, 32 + msg_len):
        text_bits += str(flat_img[i] & 1)

    # 4. Декодируем байты обратно в текст
    extracted_text = bits_to_text(text_bits)

    # 5. Сохраняем извлеченный текст в файл
    with open(output_text_path, 'w', encoding='utf-8') as file:
        file.write(extracted_text)
        
    print(f"Данные извлечены! Текст: '{extracted_text}'")
    print(f"Результат сохранен в файл: {output_text_path}")

##### Запуск и выполнение программы

In [22]:
# Основные параметры
input_image = 'A.bmp'       # Исходный контейнер
stego_image = 'B.bmp'       # Контейнер со скрытыми данными
output_text_file = 'A.txt'  # Файл для расшифрованных данных
secret_text = 'Шарибжанов'

# Запускаем встраивание
print("--- ЭТАП 1: СКРЫТИЕ ДАННЫХ ---")
embed_text(input_image, secret_text, stego_image)

# Запускаем извлечение
print("\n--- ЭТАП 2: ИЗВЛЕЧЕНИЕ ДАННЫХ ---")
extract_text(stego_image, output_text_file)

--- ЭТАП 1: СКРЫТИЕ ДАННЫХ ---
Успех! Данные встроены. Файл сохранен как: B.bmp

--- ЭТАП 2: ИЗВЛЕЧЕНИЕ ДАННЫХ ---
Данные извлечены! Текст: 'Шарибжанов'
Результат сохранен в файл: A.txt
